# ARC-AGI-3 LLM Agent v4

Ultra-fast agent with aggressive exploration.

**Key improvements:**
- Ultra-short prompt (max 500 tokens)
- Aggressive exploration (more actions, less thinking)
- Automatic new object testing
- Stuck detection and strategy change

In [ ]:
# Install dependencies
!pip install -q openai numpy pillow

import os
os.environ['KAGGLE_IS_COMPETITION_RERUN'] = '1'

In [ ]:
# Grid utilities
"""Grid formatting and segmentation utilities for ARC-AGI-3 LLM agent.

Derived from the Duck harness (MIT license, Tufa Labs).
Simplified for use with the ARC-AGI-3-Agents framework.
"""
from __future__ import annotations

import hashlib
from typing import Any, Sequence

# ARC color palette: index -> single-char label
ARC_COLOR_CHARS = "WwgGcBMPRbSYOrNp"
ARC_COLOR_LEGEND = (
    "W=white, w=light gray, g=gray, G=dark gray, c=charcoal, B=black, "
    "M=magenta, P=pink, R=red, b=blue, S=sky blue, Y=yellow, O=orange, "
    "r=dark red, N=light green, p=purple"
)

# ARC color palette: index -> RGB
ARC_COLOR_MAP: dict[int, tuple[int, int, int]] = {
    0: (255, 255, 255), 1: (204, 204, 204), 2: (153, 153, 153),
    3: (102, 102, 102), 4: (51, 51, 51),    5: (0, 0, 0),
    6: (229, 58, 163),  7: (255, 123, 204), 8: (249, 60, 49),
    9: (30, 147, 255),  10: (136, 216, 241), 11: (255, 220, 0),
    12: (255, 133, 27), 13: (146, 18, 49),  14: (79, 204, 48),
    15: (163, 86, 214),
}


def format_grid_ascii(grid: Sequence[Sequence[int]]) -> str:
    """Render a numeric grid as ASCII art using ARC color characters."""
    if not grid:
        return "(empty grid)"
    lines = []
    for row in grid:
        chars = []
        for v in row:
            chars.append(ARC_COLOR_CHARS[max(0, min(15, int(v)))])
        lines.append("".join(chars))
    return "\n".join(lines)


def grid_to_png_bytes(grid: Sequence[Sequence[int]], upscale: int = 4) -> bytes:
    """Render a numeric grid as a PNG image (no PIL dependency — pure bytes)."""
    try:
        from PIL import Image
        import io
    except ImportError:
        return b""
    
    rows = len(grid)
    cols = max((len(r) for r in grid), default=0) if rows else 0
    if rows <= 0 or cols <= 0:
        return b""
    
    image = Image.new("RGB", (cols, rows), ARC_COLOR_MAP[0])
    pixels = image.load()
    for row_idx, row in enumerate(grid):
        for col_idx in range(cols):
            value = row[col_idx] if col_idx < len(row) else 0
            pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
    
    if upscale > 1:
        image = image.resize((cols * upscale, rows * upscale), Image.Resampling.NEAREST)
    
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


# ---------------------------------------------------------------------------
# Segmentation (connected-component analysis)
# ---------------------------------------------------------------------------

_ORTH = ((-1, 0), (1, 0), (0, -1), (0, 1))
_CW = ((-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1))
_CW_INDEX = {off: i for i, off in enumerate(_CW)}


def _trace_outer_contour(cells, start):
    """Moore-neighbour trace of outer perimeter, clockwise."""
    if len(cells) == 1:
        return [start]
    contour = [start]
    b = start
    prev = (start[0], start[1] - 1)
    second = None
    for _ in range(8 * len(cells) + 16):
        idx = _CW_INDEX[(prev[0] - b[0], prev[1] - b[1])]
        nxt = None
        for k in range(1, 9):
            off = _CW[(idx + k) % 8]
            cand = (b[0] + off[0], b[1] + off[1])
            if cand in cells:
                nxt = cand
                back = _CW[(idx + k - 1) % 8]
                new_prev = (b[0] + back[0], b[1] + back[1])
                break
        if nxt is None:
            break
        if second is None:
            second = nxt
        elif b == start and nxt == second:
            break
        contour.append(nxt)
        prev, b = new_prev, nxt
    if len(contour) > 1 and contour[-1] == contour[0]:
        contour.pop()
    return contour


def _corner_points(contour):
    """Reduce contour to direction-change points."""
    if len(contour) <= 2:
        return list(contour)
    m = len(contour)
    corners = []
    for i in range(m):
        prev_c, cur, nxt = contour[i - 1], contour[i], contour[(i + 1) % m]
        d_in = (cur[0] - prev_c[0], cur[1] - prev_c[1])
        d_out = (nxt[0] - cur[0], nxt[1] - cur[1])
        if d_in != d_out:
            corners.append(cur)
    return corners


def _object_hash(cells, color):
    """Translation-invariant signature of an object."""
    min_r = min(r for r, _ in cells)
    min_c = min(c for _, c in cells)
    norm = sorted((r - min_r, c - min_c) for r, c in cells)
    payload = repr((color, norm)).encode()
    return hashlib.sha1(payload).hexdigest()[:16]


def segment_grid(grid: Sequence[Sequence[int]]) -> dict[str, Any]:
    """Segment a grid into connected components.
    
    Returns dict with:
      - nodes: list of {id, color, hash, pixels, boundary, children}
      - adjacency_list: list of [i, j] pairs
    """
    height = len(grid)
    width = len(grid[0]) if height else 0
    if not height or not width:
        return {"nodes": [], "adjacency_list": []}
    
    # Connected components via flood fill
    comp_id = [[-1] * width for _ in range(height)]
    components = []
    
    for sr in range(height):
        for sc in range(width):
            if comp_id[sr][sc] != -1:
                continue
            value = grid[sr][sc]
            cid = len(components)
            cells = set()
            stack = [(sr, sc)]
            while stack:
                r, c = stack.pop()
                if (r, c) in cells:
                    continue
                if 0 <= r < height and 0 <= c < width and grid[r][c] == value and comp_id[r][c] == -1:
                    cells.add((r, c))
                    comp_id[r][c] = cid
                    for dr, dc in _ORTH:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < height and 0 <= nc < width and comp_id[nr][nc] == -1 and grid[nr][nc] == value:
                            stack.append((nr, nc))
            components.append({"value": value, "cells": cells, "start": (sr, sc)})
    
    # Build nodes
    nodes = []
    for cid, comp in enumerate(components):
        color_char = ARC_COLOR_CHARS[max(0, min(15, comp["value"]))]
        obj_hash = _object_hash(comp["cells"], comp["value"])
        contour = _trace_outer_contour(comp["cells"], comp["start"])
        boundary = _corner_points(contour)
        nodes.append({
            "id": cid,
            "color": color_char,
            "hash": obj_hash,
            "pixels": len(comp["cells"]),
            "boundary": [[r, c] for r, c in boundary[:12]],  # limit size
            "children": [],
        })
    
    # Compute containment (children)
    for outer_id, outer_comp in enumerate(components):
        outer_cells = outer_comp["cells"]
        for inner_id, inner_comp in enumerate(components):
            if inner_id == outer_id:
                continue
            # Check if inner is fully enclosed by outer
            inner_cells = inner_comp["cells"]
            if all(any((r + dr, c + dc) in outer_cells for dr, dc in _ORTH)
                   for r, c in inner_cells):
                # Check no path from inner to grid edge without crossing outer
                # Simple check: all neighbors of inner cells are either inner or outer
                is_child = True
                for r, c in inner_cells:
                    for dr, dc in _ORTH:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < height and 0 <= nc < width:
                            if comp_id[nr][nc] != outer_id and comp_id[nr][nc] != inner_id:
                                is_child = False
                                break
                    if not is_child:
                        break
                if is_child:
                    nodes[outer_id]["children"].append(inner_id)
    
    # Compute adjacency
    adj_set = set()
    for r in range(height):
        for c in range(width):
            cid = comp_id[r][c]
            for dr, dc in _ORTH:
                nr, nc = r + dr, c + dc
                if 0 <= nr < height and 0 <= nc < width:
                    nid = comp_id[nr][nc]
                    if cid != nid:
                        pair = (min(cid, nid), max(cid, nid))
                        adj_set.add(pair)
    
    return {
        "nodes": nodes,
        "adjacency_list": sorted(adj_set),
    }


def compact_segmentation_summary(seg: dict, max_nodes: int = 20) -> str:
    """Create a compact text summary of segmentation for LLM context."""
    nodes = seg.get("nodes", [])
    adj = seg.get("adjacency_list", [])
    
    lines = [f"Objects: {len(nodes)}, Adjacencies: {len(adj)}"]
    for node in nodes[:max_nodes]:
        boundary_str = ",".join(f"{r},{c}" for r, c in node["boundary"][:4])
        children_str = f", children=[{','.join(map(str, node['children']))}]" if node["children"] else ""
        lines.append(
            f"  obj#{node['id']}: color={node['color']}, size={node['pixels']}px, "
            f"hash={node['hash'][:8]}, corners=[{boundary_str}]{children_str}"
        )
    if len(nodes) > max_nodes:
        lines.append(f"  ... and {len(nodes) - max_nodes} more objects")
    return "\n".join(lines)


In [ ]:
# LLM policy v4
"""
LLM Policy v4 for ARC-AGI-3

Ultra-fast agent with aggressive exploration:
- Max 500 token prompt
- Aggressive exploration (more actions, less thinking)
- Automatic new object testing
- Stuck detection and strategy change
"""

import os
import json
import time
from typing import Any
from dataclasses import dataclass
from openai import OpenAI

from grid_utils import format_grid_ascii, segment_grid


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

@dataclass
class Config:
    max_tokens: int = 1024  # Shorter responses
    temperature: float = 0.7  # More exploration
    timeout: float = 30.0  # Faster timeout
    context_window: int = 8192  # Smaller context
    max_actions: int = 400
    stuck_threshold: int = 5  # Change strategy after 5 no-change actions


# ---------------------------------------------------------------------------
# System prompt (ULTRA SHORT - max 500 tokens)
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """Play grid game. Act fast.

RULES:
- 64x64 grid, colored cells
- Actions: UP/DOWN/LEFT/RIGHT, MOUSE(row,col)
- GAME_OVER=reset, WIN=done
- Decreasing counter = STEP LIMIT (not timer)

STRATEGY:
1. MOVE immediately - don't think
2. If stuck 5x, try NEW approach
3. New object? CLICK it NOW
4. Track: what changes grid = good

GOALS: cover targets, reach positions, copy patterns, reduce pieces, place objects

Output: [{"action":"RIGHT"},{"action":"DOWN"}]
MOUSE: {"action":"MOUSE","row":5,"col":10}
"""


# ---------------------------------------------------------------------------
# State tracking
# ---------------------------------------------------------------------------

@dataclass
class State:
    action_count: int = 0
    tokens_used: int = 0
    last_grid_hash: str = ""
    stuck_count: int = 0
    strategy: str = "explore"  # explore, test_new, avoid_hud
    tested_positions: set = None
    
    def __post_init__(self):
        if self.tested_positions is None:
            self.tested_positions = set()


# ---------------------------------------------------------------------------
# Policy
# ---------------------------------------------------------------------------

class LLMPolicy:
    """LLM-based policy with aggressive exploration"""
    
    def __init__(self, config: Config = None):
        self.config = config or Config()
        self.client = OpenAI(
            base_url=os.environ.get("LLM_BASE_URL", "http://localhost:9000/v1"),
            api_key=os.environ.get("OPENAI_API_KEY", "dummy"),
            timeout=self.config.timeout
        )
        self.state = State()
        self.history = []
    
    def get_action(self, frame: Any) -> dict:
        """Get next action from LLM"""
        
        # Check if done
        if self.state.action_count >= self.config.max_actions:
            return {"action": "RESET"}
        
        # Extract state
        grid = self._get_grid(frame)
        grid_ascii = format_grid_ascii(grid)
        available_actions = self._get_available_actions(frame)
        
        # Detect stuck
        grid_hash = hash(grid_ascii)
        if grid_hash == self.state.last_grid_hash:
            self.state.stuck_count += 1
            if self.state.stuck_count >= self.config.stuck_threshold:
                self.state.strategy = "avoid_hud"
                self.state.stuck_count = 0
        else:
            self.state.stuck_count = 0
            self.state.last_grid_hash = grid_hash
        
        # Detect new objects (simple heuristic: new colors)
        new_colors = self._detect_new_objects(grid)
        if new_colors and self.state.strategy != "avoid_hud":
            self.state.strategy = "test_new"
        
        # Build prompt
        user_prompt = self._build_prompt(grid_ascii, available_actions)
        
        # Call LLM
        try:
            response = self.client.chat.completions.create(
                model=os.environ.get("LLM_MODEL", "gpt-4o-mini"),
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=self.config.max_tokens,
                temperature=self.config.temperature
            )
            
            content = response.choices[0].message.content
            self.state.tokens_used += response.usage.total_tokens
            
            # Parse action
            action = self._parse_action(content)
            
        except Exception as e:
            print(f"LLM error: {e}")
            action = self._fallback_action(available_actions)
        
        # Execute
        self.state.action_count += 1
        self.history.append(action)
        
        return action
    
    def _build_prompt(self, grid_ascii: str, available_actions: list) -> str:
        """Build short user prompt"""
        
        strategy_hint = ""
        if self.state.strategy == "test_new":
            strategy_hint = "\nNEW OBJECTS detected - CLICK them!"
        elif self.state.strategy == "avoid_hud":
            strategy_hint = "\nSTUCK - try DIFFERENT approach, avoid edges"
        
        prompt = f"""Grid:
{grid_ascii}

Actions: {', '.join(available_actions)}
Strategy: {self.state.strategy}{strategy_hint}

Next 3-5 actions:"""
        
        return prompt
    
    def _parse_action(self, content: str) -> dict:
        """Parse action from LLM response"""
        
        try:
            # Try to parse JSON
            actions = json.loads(content)
            if isinstance(actions, list) and len(actions) > 0:
                return actions[0]
            elif isinstance(actions, dict):
                return actions
        except:
            pass
        
        # Fallback: extract action name
        for action_name in ["UP", "DOWN", "LEFT", "RIGHT"]:
            if action_name in content.upper():
                return {"action": action_name}
        
        # Default
        return {"action": "RIGHT"}
    
    def _fallback_action(self, available_actions: list) -> dict:
        """Fallback action when LLM fails"""
        
        # Cycle through directions
        directions = ["UP", "RIGHT", "DOWN", "LEFT"]
        idx = self.state.action_count % len(directions)
        
        for direction in directions[idx:] + directions[:idx]:
            if direction in available_actions:
                return {"action": direction}
        
        return {"action": available_actions[0] if available_actions else "RIGHT"}
    
    def _get_grid(self, frame: Any) -> list:
        """Extract grid from frame"""
        if hasattr(frame, 'grid'):
            return frame.grid
        elif isinstance(frame, dict) and 'grid' in frame:
            return frame['grid']
        return []
    
    def _get_available_actions(self, frame: Any) -> list:
        """Get available actions from frame"""
        if hasattr(frame, 'available_actions'):
            return frame.available_actions
        elif isinstance(frame, dict) and 'available_actions' in frame:
            return frame['available_actions']
        return ["UP", "DOWN", "LEFT", "RIGHT", "MOUSE"]
    
    def _detect_new_objects(self, grid: list) -> list:
        """Detect new objects in grid (simple color-based heuristic)"""
        
        # Get unique colors
        colors = set()
        for row in grid:
            for cell in row:
                if cell != 0:  # Ignore background
                    colors.add(cell)
        
        # Simple heuristic: if we see new colors, there are new objects
        # (This is very basic - real implementation would track across frames)
        return list(colors)[:3]  # Return up to 3 new colors
    
    def reset(self):
        """Reset state for new game"""
        self.state = State()
        self.history = []


In [ ]:
# LLM agent
"""ARC-AGI-3 Agent adapter for the LLM-based policy.

Plugs LLMPolicy into the ARC-AGI-3-Agents reference framework.
The agent calls the LLM to get actions, with anti-loop detection
and action batching for efficiency.
"""
from __future__ import annotations

import os
import sys
import time
from threading import RLock
from typing import Any

# Ensure imports work both in competition and locally
_dir = os.path.dirname(os.path.abspath(__file__))
_parent = os.path.dirname(_dir)
for p in (_dir, _parent):
    if p not in sys.path:
        sys.path.insert(0, p)

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

try:
    from .llm_policy import LLMPolicy, LLMConfig, FrameInfo
    from .grid_utils import format_grid_ascii
except ImportError:
    from llm_policy import LLMPolicy, LLMConfig, FrameInfo
    from grid_utils import format_grid_ascii


def _extract_grid(frame_data: FrameData) -> tuple[tuple[int, ...], ...]:
    """Extract the numeric grid from a FrameData object."""
    # Try multiple paths to get the grid data
    for attr_path in [
        lambda f: f.raw_state.data,
        lambda f: f.raw_state.planes[0] if f.raw_state.planes else None,
        lambda f: f.planes[0] if hasattr(f, 'planes') and f.planes else None,
    ]:
        try:
            data = attr_path(frame_data)
            if data is None:
                continue
            if hasattr(data, 'tolist'):
                rows = data.tolist()
            else:
                rows = list(data)
            if rows and rows[0]:
                return tuple(tuple(int(c) for c in row) for row in rows)
        except Exception:
            continue
    # Absolute fallback
    return ((0,) * 64,) * 64


def _frame_to_info(frame_data: FrameData) -> FrameInfo:
    """Convert FrameData to our FrameInfo."""
    grid = _extract_grid(frame_data)
    
    # State name
    try:
        state = frame_data.state
        state_name = state.name if hasattr(state, 'name') else str(state)
    except Exception:
        state_name = "PLAYING"
    
    # Levels completed
    try:
        levels_completed = int(frame_data.levels_completed)
    except Exception:
        levels_completed = 0
    
    # Available actions
    available = []
    try:
        for act_id in frame_data.available_actions:
            try:
                act = GameAction.from_id(int(act_id))
                if act.name != "RESET":
                    available.append(act.name)
            except Exception:
                pass
    except Exception:
        available = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "ACTION7"]
    
    # Step number
    try:
        step = int(frame_data.step)
    except Exception:
        step = 0
    
    return FrameInfo(
        grid=grid,
        step=step,
        level=levels_completed + 1,
        state_name=state_name,
        levels_completed=levels_completed,
        available_actions=available,
    )


class LLMAgent(Agent):
    """LLM-powered agent for ARC-AGI-3."""
    
    _ACTION_LOCK = RLock()
    
    ACTION_BUDGET = 400
    MAX_ACTIONS = ACTION_BUDGET
    
    # Soft time limit per game (seconds)
    # 110 games × 4 min = 440 min ≈ 7.3 hours — fits in 9h with margin
    GAME_TIME_LIMIT = 240
    
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.config = LLMConfig.from_env()
        self.policy = LLMPolicy(self.config)
        self._holds_action_lock = False
        self._game_start_time = time.time()
        self._actions_in_game = 0
        self._first_action = True
    
    @property
    def name(self) -> str:
        return f"{super().name}.llm-v2"
    
    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Stop conditions: win, budget, or time limit."""
        if self._actions_in_game >= self.ACTION_BUDGET:
            return True
        try:
            if latest_frame.state is GameState.WIN:
                return True
        except Exception:
            pass
        if time.time() - self._game_start_time > self.GAME_TIME_LIMIT:
            return True
        return False
    
    @staticmethod
    def _to_game_action(proposal: dict) -> GameAction:
        """Convert policy output to GameAction enum."""
        action_name = proposal.get("name", "RESET")
        
        try:
            action = GameAction[action_name]
        except KeyError:
            action = GameAction.RESET
        
        if action.is_complex():
            x = proposal.get("x", 0) or 0
            y = proposal.get("y", 0) or 0
            action.set_data({"x": max(0, min(63, int(x))), "y": max(0, min(63, int(y)))})
        else:
            action.set_data({})
        
        action.reasoning = proposal.get("reasoning", {})
        return action
    
    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        """Main entry point called by the framework each turn."""
        self._ACTION_LOCK.acquire()
        self._holds_action_lock = True
        
        try:
            frame_info = _frame_to_info(latest_frame)
            
            # Initialize on first call
            if self._first_action:
                self._first_action = False
                self._game_start_time = time.time()
                self.policy.reset_game()
            
            # Auto-reset on GAME_OVER
            if frame_info.state_name == "GAME_OVER":
                action = GameAction.RESET
                action.set_data({})
                action.reasoning = {"auto_reset": True, "game_over": True}
                return action
            
            # Call the LLM policy
            proposal = self.policy.choose_action(frame_info)
            self._actions_in_game += 1
            
            return self._to_game_action(proposal)
            
        except Exception as e:
            import traceback
            print(f"[LLMAgent] Error: {e}")
            traceback.print_exc()
            action = GameAction.RESET
            action.set_data({})
            action.reasoning = {"error": str(e)}
            return action
        finally:
            if self._holds_action_lock:
                self._holds_action_lock = False
                self._ACTION_LOCK.release()
    
    def take_action(self, action: GameAction):  # type: ignore[override]
        """Execute action and record the result."""
        try:
            frame = super().take_action(action)
            if frame is not None:
                frame.action_input = ActionInput(
                    id=action,
                    data=action.action_data.model_dump(),
                    reasoning=getattr(action, "reasoning", None),
                )
            return frame
        finally:
            if getattr(self, "_holds_action_lock", False):
                self._holds_action_lock = False
                self._ACTION_LOCK.release()
    
    def finalize_diagnostics(self) -> None:
        """Log final stats for this game."""
        elapsed = time.time() - self._game_start_time
        print(f"[LLMAgent] Game done: "
              f"{self._actions_in_game} actions in {elapsed:.0f}s, "
              f"{self.policy.total_tokens} tokens ({self.policy.total_calls} calls, "
              f"{self.policy.client.total_time:.0f}s LLM time), "
              f"{len(self.policy.tracker.unique_states_seen)} unique states, "
              f"{self.policy.tracker.game_over_count} deaths")


In [ ]:
# Run benchmark
import arc_agi
from arc_agi.benchmark import Benchmark

benchmark = Benchmark()
benchmark.run()